# 2.1 需求分布探索

先把"需求量"这个词定死：某个类目某个月的正数量合计，退货行不计入。
然后看各类目逐月的需求量，判断有没有季节性——有的话，4.1 才值得做季节特征。

In [1]:
import sys
from pathlib import Path

import pandas as pd

import dsflow

ROOT = Path.cwd()
while not (ROOT / "dsflow.yaml").is_file():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from demo_lib import out, season_table

OUT = out("2.1")
CLEAN = out("1.2") / "orders_clean.parquet"
run = dsflow.start_run("2.1", project=ROOT, hypothesis="各类目的月度需求量有季节性，高峰月份各不相同")

run.log_input(CLEAN, name="orders_clean")
clean = pd.read_parquet(CLEAN)
demand = clean[~clean["是否退货"]]
monthly = (demand.groupby(["类目", "下单月份"], as_index=False)["数量"].sum()
           .rename(columns={"下单月份": "月份", "数量": "需求量"}))
monthly.to_csv(OUT / "monthly_by_category.csv", index=False)
print(f"退货行 {int(clean['是否退货'].sum()):,} 行不计入需求量")
print(f"月度需求量表：{len(monthly)} 行（{monthly['类目'].nunique()} 个类目 × {monthly['月份'].nunique()} 个月）")
monthly.head(6)


退货行 361 行不计入需求量
月度需求量表：72 行（3 个类目 × 24 个月）


,类目,月份,需求量
0,MRO工业品,2024-07,6719
1,MRO工业品,2024-08,7099
2,MRO工业品,2024-09,5933
3,MRO工业品,2024-10,8996
4,MRO工业品,2024-11,5869
5,MRO工业品,2024-12,6271


In [2]:
table = season_table(monthly)
season = pd.DataFrame([{"类目": c, "月": int(m), "季节系数": v} for c, d in table.items() for m, v in d.items()])
season = season.sort_values(["类目", "月"]).reset_index(drop=True)
season.to_csv(OUT / "seasonality.csv", index=False)
peaks = season.loc[season.groupby("类目")["季节系数"].idxmax()]
print(peaks.to_string(index=False))


    类目  月   季节系数
MRO工业品 10 1.3070
办公通用物资  3 1.1769
  电力物资 12 1.4864


In [3]:
run.log_metrics({f"{r.类目}_高峰月季节系数": r.季节系数 for r in peaks.itertuples()})
run.log_metrics({"退货行不计入需求量": int(clean["是否退货"].sum())})
run.log_artifact(OUT / "monthly_by_category.csv", purpose="各类目月度需求量", kind="table")
run.log_artifact(OUT / "seasonality.csv", purpose="各类目 12 个自然月的季节系数", kind="table")
conclusion = "；".join(f"{r.类目} 高峰在 {r.月} 月（季节系数 {r.季节系数}）" for r in peaks.itertuples())
run.set_conclusion(conclusion, validity="有效")
run.end()
print(conclusion)


MRO工业品 高峰在 10 月（季节系数 1.307）；办公通用物资 高峰在 3 月（季节系数 1.1769）；电力物资 高峰在 12 月（季节系数 1.4864）
